### This notebook collects links and information from a given website url and generates a business brochure. 

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI



In [2]:
#only needed in Jupyter notebook
!uv pip install nest_asyncio

Using Python 3.13.18 environment at: /home/dave/Documents/Courses/aiengr
Checked 1 package in 11ms


In [3]:
# Only needed in Jupyter
import nest_asyncio
nest_asyncio.apply()

In [4]:
#Initialize LLM and load environment variables

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

#verify the key

if not api_key:
    print("No API key found")
elif not api_key.startswith("sk-proj-"):
    print("Invalid API key format")
elif api_key.strip() != api_key:
    print("An API key should not have leading or trailing whitespace")
else:
    print("API key is valid")

# name the model to use (note: this should be a variable in a future edition)
MODEL = 'gpt-6-luna'
openai = OpenAI()




API key is valid


The next cells will complete the following steps:
1. structure a system prompt as a constant
2. create a user promption function to allow changing of the entered URL to be used as the basis of the brochure
3. create a function to select the relevant links from the URL given by user input in step 2
4. Create a function to stich together the page contents and relevant links (fetch_page_and_all_relevant_links)
5. Create a brochure system prompt to instuct the LLM how to create the brochure
6. Create a brochure user prompt function for the user to enter a company name and URL
7. Create a function to generate the brochure (create_brochure)
8. Creat an updated function of item 6 to stream the results onto the screen



In [5]:
# 1. Structure a system prompt as a constant

link_system_prompt = """ 
You are provided with a list of links found on a webpage. 
You are able to decide which of the links would be most relevant to include in a brochure about the company, 
such as link to an About page, a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about},
        {"type": "careers page", "url": "https://full.url/goes/here/careers"}
    ]
}
"""

In [6]:
# 2  create a user promption function to allow changing of the entered URL to be used as the basis of the brochure

def get_links_user_prompt(url):
    user_prompt = f""" 
    Here is a list of links on the website {url} - 
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https url in JSON format. 
    Do not include Terms of Service or Privacy Policy email links. 

    """

    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))

 
    Here is a list of links on the website https://edwarddonner.com - 
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https url in JSON format. 
    Do not include Terms of Service or Privacy Policy email links. 

    #wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddonner.com/2

In [16]:
# 3. create a function to select the relevant links from the URL given by user input in step 2

def select_relevant_links(url):
    response = openai.chat.completions.create(
        model = MODEL,
        reasoning_effort="none",
        messages = [
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
    response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links



In [9]:
# Test cell for select_relevant_links function

select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'courses and curriculum',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [19]:
# 4. Create a function to stich together the page contents and relevant links (fetch_page_and_all_relevant_links)

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url, use_browser=False)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page: \n\n{contents}\n##Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        try:
            result += fetch_website_contents(link["url"], use_browser=False)
        except Exception as e:
            result += f"(Could not fetch this page: {e})"
    return result


In [22]:
# Test cell for fetch_page_and_all_relevant_links function

print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

## Landing Page: 

Home - Edward Donner

October 14, 2026
AI FDE: The Forward Deployed Engineer & Agentic AI Architect
February 17, 2026
AI Coder: Vibe Coder to Agentic Engineer – RESOURCES
January 4, 2026
AI Builder with n8n – Create Agents and Voice Agents – RESOURCES
September 15, 2025
AI Engineering MLOps Track – Deploy AI to Production – RESOURCES
##Relevant Links:


### Link: about page
About - Edward Donner

About
I’m the co-founder and CTO of
Nebula.io
. We help recruiters source, understand, engage and manage talent, using Generative AI and other forms of machine learning. Our
patented
model matches people with roles with greater accuracy and speed than previously imaginable — no keywords required. Take a look for yourself; it’s completely free to try.
Our long term goal is to help people discover their potential and pursue their reason for being, motivated by a concept called
Ikigai
. We help people find roles where they will be most fulfilled and successful; as a result, we 

In [23]:
# 5. Create a brochure system prompt to instuct the LLM how to create the brochure

brochure_system_prompt = """ 
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [27]:
# 6. Create a brochure user prompt function for the user to enter a company name and URL

def get_brochure_user_prompt(company_name, url):
    user_prompt = f""" 
You are looking at a company called: {company_name}
Here are teh contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n 
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [28]:
# Test cell for get_brochure_user_prompt function
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

' \nYou are looking at a company called: HuggingFace\nHere are teh contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n \n## Landing Page: \n\nHugging Face – The AI community building the future.\n\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nUpdated\n12 days ago\n•\n24.7k\n•\n1.71k\nUpdated\n2 days ago\n•\n11.8k\n•\n1.1k\nUpdated\n5 days ago\n•\n27.9k\n•\n1.82k\nUpdated\nabout 1 hour ago\n•\n5.84k\n•\n466\nUpdated\nabout 1 month ago\n•\n2.72M\n•\n11.3k\nBrowse 2M+ models\nSpaces\nThe Ultimate MoE Training Handbook\n🧩\n500\nA hands-on guide to training Mixture-of-Experts models\nHarness Leaderboard\n🧭\n244\nWhich coding-agent harness should you use for your model?\nWan2.2 14B Preview\n🐌\n1.46k\ngenerate a video from an

In [33]:
# 7. Create a function to generate the brochure (create_brochure)

def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="none", 
        messages = [
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))


In [38]:
#test cell for create_brochure

create_brochure("cbs", "https://www.cbsnews.com")

# CBS News

**CBS News** delivers breaking news, top stories and exclusive coverage. Its landing page highlights political reporting, including coverage of President Trump’s *60 Minutes* interview, with key takeaways and a full transcript.

## What CBS News offers
- News coverage and reporting
- Interviews and exclusive content
- A team of anchors, correspondents and contributors

## People and leadership
CBS News lists **Tom Cibrowski** as President and Executive Editor. **David Agus, M.D.** is featured as a contributor.

## Advertising and brand partnerships
**CBS News Brand Studio** provides content marketing and advertising services. The newsroom is not involved in creating Brand Studio content.

*Note: The available source pages did not provide verifiable details about company culture, customers, or current career opportunities.*

In [ ]:
# 8. Creat an updated function of item 6 to stream the results onto the screen

